# Lightningfish - HN backtest on GPU

Reproduces the Hacker News reception experiments from
[ARCHITECTURE.md](https://github.com/rajul-kk/LightningFish/blob/main/ARCHITECTURE.md)
section 10, using a free Kaggle T4.

Locally these are CPU-bound and take 6-10 hours. qwen2.5:7b Q4 (~4.7 GB) fits
entirely in a T4's 16 GB of VRAM, which makes the same work tractable.

**What gets run, in order:**

1. `hn` - submission-only seeds (the 69% karma ceiling)
2. `hn-early` - the same stories re-seeded with their first 2h of comments
3. `hn-early blind` - only the stories that drew no early comments

Order matters: the early-comments experiment reads its story ids from the
submission-only cache so the two runs are **paired** on identical events.

## 1. Setup

Set the sidebar to **Accelerator: GPU T4 x2** (or P100) and **Internet: On**
before running. Internet is required to install Ollama and pull the model.

Do not paste API keys here. This runs a local model and needs none; if you ever
want a Claude-backed run, use Kaggle **Secrets**, never an inline string.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start")

In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ollama pull {MODEL}

import requests

# Force a load so /api/ps reports placement. keep_alive=-1 pins the model so it
# is not unloaded between events (an unload/reload cycle mid-run is brutal).
requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
    timeout=600,
)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0)
    print(f"{m['name']}: size_vram={vram / 1e9:.2f} GB")
    assert vram > 0, (
        "model is on CPU, not GPU - check the accelerator is enabled. "
        "Running on CPU here is no faster than a laptop."
    )
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import os, sys

os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only. The finance/service tests pull yfinance, praw, edgar,
# fastapi, modal and psycopg, none of which an HN run touches.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

In [ ]:
# Measure real throughput before committing to a long run.
import time

from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate this: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"~{per_call:.2f}s per short LLM call")
print(f"(on a starved CPU box this was ~27s - if you see double digits, check the GPU assert above)")

## 2. Configuration

`LIGHTNINGFISH_LOCAL_TIMEOUT` bounds any single request. On GPU 120s is
generous; it exists so one wedged call cannot stall the whole run (a bug that
did exactly that on CPU before being fixed).

In [ ]:
%env LIGHTNINGFISH_MODEL=ollama:qwen2.5:7b
%env LIGHTNINGFISH_N_AGENTS=24
%env LIGHTNINGFISH_N_ROUNDS=4
%env LIGHTNINGFISH_LOCAL_TIMEOUT=120
%env PYTHONUNBUFFERED=1

## 3. Submission-only run

Pulls class-balanced settled stories (half above the high points threshold, half
below the low one) and scores the simulation against the naive karma baseline,
a single-LLM-call baseline, and the majority class.

In [ ]:
!python -m tests.integration.run_backtest hn 40 2>&1 | tee /kaggle/working/hn.log

## 4. Paired early-comments run

Same stories, now carrying every comment posted in the story's first 2 hours.
The baseline ladder gains a `naive_early` rung that predicts purely from the
comment **count**, so the simulation only earns a result by beating it - which
would mean it is reading the comment *text*, not just noticing comments exist.

The task framing changes here, so these numbers are **not** comparable to the
submission-only run above.

In [ ]:
!python -m tests.integration.run_backtest hn-early 2>&1 | tee /kaggle/working/hn_early.log

## 5. Blind subgroup

Restricted to stories that drew *no* early discussion. Locally this was 22 of
36 events (17 flop / 5 viral), where the count baseline is blind and scores
77% by guessing "flop" every time.

This is the only place the simulation can show it reads submission content.
Note the sample is small enough that the binomial test cannot register a
positive short of a very large margin - read `p_value_vs_best`, not accuracy.

In [ ]:
!python -m tests.integration.run_backtest hn-early blind 2>&1 | tee /kaggle/working/hn_blind.log

## 6. Save results

Everything under `/kaggle/working` is kept as notebook output. Saving the cache
means a later session, or your local machine, can re-score without re-fetching.

In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/*.log /kaggle/working/cache

## How to read the output

1. **Confidence first.** If `low_confidence` is set or the parse rate is under
   0.8, stop - the result reflects format failures, not dynamics.
2. **`beats_baselines` must be true for every entry**, including `single_llm`.
   Beating a naive heuristic only shows the model knows something; beating a
   single call is what shows the multi-agent machinery contributes anything.
3. **`p_value_vs_best`** is a one-sided binomial test against the *best* rung,
   not against chance. High accuracy with p > 0.05 is not a result.

See [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md)
for the full protocol.